In [1]:
!pip install -q torch-geometric

**Testing the imports and heterophily dataset loading**

In [2]:
# test that it imports and a heterophily dataset loads
import torch
from torch_geometric.datasets import WebKB

dataset = WebKB(root="data_wisconsin", name="Wisconsin")
data = dataset[0]

print("torch version:", torch.__version__)
print("dataset loaded:", dataset)
print("number of nodes:", data.num_nodes)
print("number of edges:", data.num_edges)
print("number of classes:", dataset.num_classes)
print("number of features:", dataset.num_features)

Processing...


torch version: 2.11.0+cpu
dataset loaded: wisconsin()
number of nodes: 251
number of edges: 515
number of classes: 5
number of features: 1703


Done!


**Testing the webkb dataset and computing the homophily ration**

In [3]:
import torch
from torch_geometric.datasets import WebKB
from torch_geometric.utils import homophily

# loading the heterophilous web-page graph
dataset = WebKB(root="data_wisconsin", name="Wisconsin")
data = dataset[0]

# edge homophily ratio
h = homophily(data.edge_index, data.y, method="edge")

# printing out some stats so i know it actually loaded correctly
print("edge homophily ratio:", round(h, 3))
print("nodes:", data.num_nodes, "edges:", data.num_edges, "classes:", dataset.num_classes, "features:", dataset.num_features)


edge homophily ratio: 0.196
nodes: 251 edges: 515 classes: 5 features: 1703


**Defining both the Models**
- Model A: GCN
- Model B: FAGCN

In [5]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, FAConv

# Model A: Standard 2-layer GCN model
class GCN(nn.Module):
    def __init__(self, num_features, hidden, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, hidden)
        self.conv2 = GCNConv(hidden, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)

        return x


# Model B: FAGCN for heterophily
class FAGCN(nn.Module):
    def __init__(self, num_features, hidden, num_classes, eps=0.3):
        super().__init__()

        self.lin_in = nn.Linear(num_features, hidden)

        # FAConv layers
        self.conv1 = FAConv(hidden, eps=eps)
        self.conv2 = FAConv(hidden, eps=eps)

        self.lin_out = nn.Linear(hidden, num_classes)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.5, training=self.training)

        # initial projection
        x = F.relu(self.lin_in(x))

        # save initial features for the skip connection or whatever FAConv does
        x0 = x

        x = self.conv1(x, x0, edge_index)
        x = self.conv2(x, x0, edge_index)

        x = self.lin_out(x)
        return x


# quick sanity check print
print("models loaded successfully...")

models loaded successfully...


**Training and Evaluating both the models across 10 splits**

In [6]:
from sklearn.metrics import f1_score
import numpy as np

# training function - keeping epochs at 200 cuz that usually converges fine for these small datasets
def train_one_split(model, data, train_mask, test_mask, epochs=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)

        loss = F.cross_entropy(out[train_mask], data.y[train_mask])

        loss.backward()
        optimizer.step()

    # eval time
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)

        correct = pred[test_mask] == data.y[test_mask]
        acc = int(correct.sum()) / int(test_mask.sum())

        f1 = f1_score(data.y[test_mask].cpu(), pred[test_mask].cpu(), average="macro")

    return acc, f1

# helper to run over the 10 standard splits provided in the dataset
def run_model(model_class, data, dataset, n_splits=10):
    accs = []
    f1s = []

    for i in range(n_splits):
        train_mask = data.train_mask[:, i]
        test_mask = data.test_mask[:, i]

        model = model_class(dataset.num_features, 64, dataset.num_classes)

        acc, f1 = train_one_split(model, data, train_mask, test_mask)
        accs.append(acc)
        f1s.append(f1)

    # return mean and std for reporting
    return np.mean(accs), np.std(accs), np.mean(f1s)

# let's run both models now
print("Running GCN...")
a_acc, a_std, a_f1 = run_model(GCN, data, dataset)

print("Running FAGCN...")
b_acc, b_std, b_f1 = run_model(FAGCN, data, dataset)

# print out the final results
print("------------------------------------------")
print("Model A (GCN):   acc", round(a_acc, 3), "+/-", round(a_std, 3), "| macro-F1", round(a_f1, 3))
print("Model B (FAGCN): acc", round(b_acc, 3), "+/-", round(b_std, 3), "| macro-F1", round(b_f1, 3))
print("------------------------------------------")

Running GCN...
Running FAGCN...
------------------------------------------
Model A (GCN):   acc 0.422 +/- 0.045 | macro-F1 0.303
Model B (FAGCN): acc 0.647 +/- 0.044 | macro-F1 0.468
------------------------------------------
